# Aula 1 · Notebook 06 — Pandas vs DuckDB: quando usar cada um

Objetivo: ver **na prática** a diferença de performance entre Pandas e DuckDB num dataset grande.

Spoiler: o resultado é claro, mas a moral da história é "**use a ferramenta certa para o problema**".

---

## Plano

1. Garantir que o `empresas.parquet` está baixado
2. Mesmo problema, duas soluções: pandas vs DuckDB
3. Cronometrar
4. Conclusão

Tempo estimado: 15 minutos.

In [ ]:
import urllib.request
import pathlib

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-data-2026-v1/empresas.parquet"
PATH = pathlib.Path("data/empresas.parquet")
PATH.parent.mkdir(exist_ok=True)
if not PATH.exists():
    print("Baixando...")
    urllib.request.urlretrieve(URL, PATH)
print(f"Arquivo: {PATH.stat().st_size / 1e6:.1f} MB")

## Pergunta de teste

> **Quantas empresas existem por estado?**

Pergunta simples, mas em 6,3 milhões de linhas dá pra sentir a diferença.

## Solução A — Pandas

In [ ]:
import pandas as pd
import time

t0 = time.time()
df = pd.read_parquet("data/empresas.parquet")
read_time = time.time() - t0
print(f"⏱  read_parquet: {read_time:.2f}s ({len(df):,} linhas em memória)")

t0 = time.time()
resultado_pd = df.groupby("uf").size().sort_values(ascending=False).head(10)
group_time = time.time() - t0
print(f"⏱  groupby:      {group_time:.2f}s")
print(f"📦 Memória RAM: ~{df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

resultado_pd

## Solução B — DuckDB

In [ ]:
import duckdb

t0 = time.time()
resultado_dd = duckdb.sql("""
    SELECT uf, COUNT(*) AS total
    FROM 'data/empresas.parquet'
    GROUP BY uf
    ORDER BY total DESC
    LIMIT 10
""").df()
total_time = time.time() - t0
print(f"⏱  DuckDB (read + groupby): {total_time*1000:.0f} ms")
print(f"📦 Memória RAM: quase zero (streamed)\n")

resultado_dd

## Conclusão

| Critério | Pandas | DuckDB |
|----------|--------|--------|
| Tempo para responder a pergunta | segundos | ~milissegundos |
| Carrega tudo na RAM? | sim | não |
| Sintaxe | Python | SQL |
| Integração com sklearn/scipy | nativa | converte com `.df()` |

**Quando usar cada um?**

| Use Pandas se… | Use DuckDB se… |
|---------------|----------------|
| Volume menor que ~1M linhas | Volume maior que ~1M linhas |
| Quer integrar com Scikit-learn | Quer só SQL |
| Já tem o DataFrame em mãos | Está consultando arquivos brutos |

**Você pode misturar!** `duckdb.sql(...).df()` devolve um DataFrame do Pandas. Isso é muito comum: DuckDB filtra/agrega o pesado, Pandas faz o "último km" de visualização ou modelagem.

## Mexa você mesmo

1. Faça a mesma query, mas em vez de UF, agrupe por `porte`. Compare os dois.
2. Tente carregar `empresas.parquet` com `pd.read_csv` (não vai funcionar). Por quê?

In [ ]:
# Sua vez!


---

### Próximo notebook

[07 — Bloco 3: Análise real com Olist](./07-bloco-3-olist-analise.ipynb)